In [ ]:
import os
import numpy as np

data = np.load('modelling_data.npz')
Xraw = data['Xraw']
yraw = data['yraw']
Xfit = data['Xfit']
yfit = data['yfit']
print(Xraw.shape, yraw.shape, Xfit.shape, yfit.shape)

In [ ]:
# check Xraw correlation plot
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(np.corrcoef(Xraw.T), annot=True, cmap='coolwarm')
plt.title('Correlation matrix of Xraw')
plt.show()

In [ ]:
# check Xfit correlation plot
sns.heatmap(np.corrcoef(Xfit.T), annot=True, cmap='coolwarm')
plt.title('Correlation matrix of Xfit')
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def build_model(input_dim, shapes=[128, 128, 64]):
    model = models.Sequential()
    model.add(layers.InputLayer(input_shape=(input_dim,)))
    for shape in shapes:
        model.add(layers.Dense(shape, activation='relu'))
    model.add(layers.Dense(1))  # Single linear output for regression (cross-section)
    return model


In [ ]:

# Split fitted data for Pre-Training
Xfit_train, Xfit_test, yfit_train, yfit_test = train_test_split(
    Xfit, yfit, test_size=0.2, random_state=42
)

# Split raw data for Fine-Tuning
Xraw_train, Xraw_test, yraw_train, yraw_test = train_test_split(
    Xraw, yraw, test_size=0.2, random_state=42
)

# 3. Feature Scaling (CRITICAL)
print("Scaling features...")
scaler = StandardScaler()
# Fit the scaler on the fitted training data, then apply to EVERYTHING else
Xfit_train_scaled = scaler.fit_transform(Xfit_train)
Xfit_test_scaled = scaler.transform(Xfit_test)

Xraw_train_scaled = scaler.transform(Xraw_train)
Xraw_test_scaled = scaler.transform(Xraw_test)

# 4. Build the Neural Network Architecture

model = build_model(Xfit_train_scaled.shape[1])

# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# =====================================================================
# PHASE 1: PRE-TRAINING (Learning the Physics)
# =====================================================================
print("\n--- PHASE 1: Pre-training on Fitted Data ---")
# Standard learning rate for initial training
model.compile(optimizer=optimizers.Adam(learning_rate=0.001), loss='mse')

history_fit = model.fit(
    Xfit_train_scaled, yfit_train,
    validation_data=(Xfit_test_scaled, yfit_test),
    epochs=100,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

# =====================================================================
# PHASE 2: FINE-TUNING (Adjusting to Raw Reality)
# =====================================================================
print("\n--- PHASE 2: Fine-tuning on Raw Data ---")
# Lower the learning rate drastically so we don't destroy the physics we just learned
model.compile(optimizer=optimizers.Adam(learning_rate=0.0001), loss='mse')

history_raw = model.fit(
    Xraw_train_scaled, yraw_train,
    validation_data=(Xraw_test_scaled, yraw_test),
    epochs=50,
    batch_size=64, # Smaller batch size for smaller dataset
    callbacks=[early_stop],
    verbose=1
)

# =====================================================================
# EVALUATION
# =====================================================================
print("\n--- Final Evaluation on Raw Test Set ---")
loss = model.evaluate(Xraw_test_scaled, yraw_test, verbose=0)
print(f"Final Mean Squared Error (MSE) on unseen raw data: {loss:.6e}")

# Save the final model so you don't have to retrain it later
model.save("cross_section_transfer_model.keras")
print("Model saved to 'cross_section_transfer_model.keras'")